In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import numpy as np
import pandas as pd
import sympy
import sympy.printing

In [3]:
# Read data
read_data_005=pd.read_csv("D:/Man/Earth and Environment/Manchester_data003-008/005_manchester_processed.csv")
read_data_005=read_data_005.sort_values(by='time').reset_index(drop=True)

learning_data_005=read_data_005[(read_data_005['year']>=2006)&(read_data_005['year']<=2050)].copy()
learning_data_005=learning_data_005.drop(columns=['lat','lon','date'],errors='ignore')

learning_data_005

,time,TREFMXAV_U,FLNS,FSNS,PRECT,PRSN,QBOT,TREFHT,UBOT,VBOT,year,month,day,dayofyear
0,2006-01-02,11.15260,8.737288,7.855509,1.544200e-07,4.051599e-16,0.005373,6.07195,2.303055,4.502803,2006,1,2,2
1,2006-01-03,13.01327,6.686464,7.501073,7.784098e-08,0.000000e+00,0.007595,11.04843,4.657475,3.158464,2006,1,3,3
2,2006-01-04,13.16030,27.445148,12.188718,4.851411e-08,7.075068e-18,0.005667,9.27023,5.083677,5.835154,2006,1,4,4
3,2006-01-05,14.85882,10.443632,4.354691,1.091676e-07,1.429017e-14,0.007362,11.82208,3.029053,8.031938,2006,1,5,5
4,2006-01-06,10.78576,70.927300,31.532597,2.531008e-09,6.418103e-18,0.004160,7.14828,3.783906,6.986506,2006,1,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15811,2050-12-27,10.58290,19.547508,15.106448,2.213155e-08,6.438513e-15,0.005779,8.01800,0.623059,4.167736,2050,12,361,361
15812,2050-12-28,14.16284,18.568079,13.015922,3.699803e-08,9.786361e-23,0.006798,9.50298,-1.815245,4.800050,2050,12,362,362
15813,2050-12-29,16.11770,15.843533,8.881753,2.557600e-08,1.473843e-22,0.008161,13.01970,-2.800902,7.744351,2050,12,363,363
15814,2050-12-30,13.72466,21.278680,9.996034,4.276372e-08,1.328554e-12,0.006569,10.77773,-1.253943,6.456124,2050,12,364,364


In [4]:
from sklearn.preprocessing import StandardScaler

features = ['FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']


X_005 = learning_data_005[features].copy()
y_005 = learning_data_005['TREFMXAV_U'].values

X_005['month_sin'] = np.sin(2 * np.pi * learning_data_005['month'] / 12)
X_005['month_cos'] = np.cos(2 * np.pi * learning_data_005['month'] / 12)

X_005['day_sin'] = np.sin(2 * np.pi * learning_data_005['dayofyear'] / 365.25)
X_005['day_cos'] = np.cos(2 * np.pi * learning_data_005['dayofyear'] / 365.25)

train_mask = learning_data_005['year'] <= 2040

X_train_raw = X_005[train_mask].values
y_train_raw = y_005[train_mask]

X_val_raw = X_005[~train_mask].values
y_val_raw = y_005[~train_mask]

scaler_X = StandardScaler()
X_train_raw=scaler_X.fit_transform(X_train_raw)
X_val_raw=scaler_X.transform(X_val_raw)

print(f"Training set: {len(X_train_raw)}")
print(f"Validation set: {len(X_val_raw)}")

X_train_t = torch.FloatTensor(X_train_raw)
y_train_t = torch.FloatTensor(y_train_raw).view(-1, 1)

X_val_t = torch.FloatTensor(X_val_raw)
y_val_t = torch.FloatTensor(y_val_raw).view(-1, 1)

Training set: 12311
Validation set: 3505


In [5]:
# EarlyStopping
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0):
        self.patience = patience 
        self.verbose = verbose
        self.counter = 0   
        self.best_score = None 
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        self.best_state_dict = model.state_dict()
        self.val_loss_min = val_loss

# MLP
Hidden Layer1=64

Hidden Layer2=32

learning rate=0.001

epoch=200

patience=15

In [8]:
class TemperatureMLP(nn.Module):
    def __init__(self, input_channels):
        super(TemperatureMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_channels, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

In [14]:
import torch.nn.functional as F
from sklearn.metrics import max_error,r2_score

# Training
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TemperatureMLP(input_channels=X_train_t.shape[1]).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=15, factor=0.5)
criterion = nn.L1Loss()
early_stopping = EarlyStopping(patience=15, verbose=True) 

X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)


for epoch in range(200):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_mae = criterion(val_preds, y_val_t).item()
    
    scheduler.step(val_mae)
    early_stopping(val_mae, model)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss {total_loss/len(train_loader):.4f}, Val MAE {val_mae:.4f}")
    
    if early_stopping.early_stop:
        break

model.load_state_dict(early_stopping.best_state_dict)
model.eval()
with torch.no_grad():
    final_preds_tensor = model(X_val_t)
    y_true = y_val_t.cpu().numpy().flatten()
    y_pred = final_preds_tensor.cpu().numpy().flatten()

    mae = criterion(final_preds_tensor, y_val_t).item()
    mse = F.mse_loss(final_preds_tensor, y_val_t).item()
    rmse = np.sqrt(F.mse_loss(final_preds_tensor, y_val_t).item())
    max_err = max_error(y_true, y_pred)
    r2=r2_score(y_true, y_pred)


print("\n" + "="*30)
print(f"Point Prediction Result (2041-2050):")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"{'Max Error':<15}: {max_err:.4f}")
print(f"{'R-Squared':<15}: {r2:.4f}")
print("="*30)

EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
Epoch 10: Train Loss 1.0095, Val MAE 0.6158
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
Epoch 20: Train Loss 0.7925, Val MAE 0.5726
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 6 out of 15
EarlyStopping counter: 7 out of 15
Epoch 30: Train Loss 0.6971, Val MAE 0.5637
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
Epoch 40: Train Loss 0.6512,

# Transformer
One encoder layer

d_model=32

learning rate=0.001

epoch=200

patience=15

In [30]:
class PointTransformer(nn.Module):
    def __init__(self, input_channels, d_model=32, nhead=4, num_layers=1, dropout=0.1):
        super(PointTransformer, self).__init__()
        
        self.input_fc = nn.Linear(input_channels, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, 
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.output_fc = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1) 
        x = self.input_fc(x)
        x = self.transformer_encoder(x)
        x = x.squeeze(1)
        return self.output_fc(x)

In [31]:
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_channels = X_train_t.shape[1]
model = PointTransformer(input_channels).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=15, factor=0.5)
criterion = nn.L1Loss()
early_stopping = EarlyStopping(patience=15, verbose=True) 

X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)

for epoch in range(200):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_mae = criterion(val_preds, y_val_t).item()
    
    scheduler.step(val_mae)
    early_stopping(val_mae, model)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss {total_loss/len(train_loader):.4f}, Val MAE {val_mae:.4f}")
    
    if early_stopping.early_stop:
        break


model.load_state_dict(early_stopping.best_state_dict)
model.eval()

with torch.no_grad():
    final_preds_tensor = model(X_val_t)
    y_true = y_val_t.cpu().numpy().flatten()
    y_pred = final_preds_tensor.cpu().numpy().flatten()
    
    mae = criterion(final_preds_tensor, y_val_t).item()
    rmse = np.sqrt(F.mse_loss(final_preds_tensor, y_val_t).item())
    max_err = max_error(y_true, y_pred)
    r2=r2_score(y_true, y_pred)

print("\n" + "="*40)
print(f"Transformer Point Result (2041-2050):")
print(f"{'MAE':<15}: {mae:.4f}")
print(f"{'RMSE':<15}: {rmse:.4f}")
print(f"{'Max Error':<15}: {max_err:.4f}")
print(f"{'R-Squared':<15}: {r2:.4f}")
print("="*40)

EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
Epoch 10: Train Loss 1.4899, Val MAE 0.6907
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
Epoch 20: Train Loss 1.4218, Val MAE 0.8630
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 6 out of 15
EarlyStopping counter: 7 out of 15
EarlyStopping counter: 8 out of 15
EarlyStopping counter: 9 out of 15
EarlyStopping counter: 10 out of 15
EarlyStopping counter: 11 out of 15
EarlyStopping counter: 12 out of 15
Epoch 30: Train Loss 1.3302, Val MAE 0.5787
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 

# LSTM
hidden_size=32

learning rate=0.001

epoch=200

patience=15

In [32]:
class PointLSTM(nn.Module):
    def __init__(self, input_channels, hidden_size=32, num_layers=1, dropout=0.2):
        super(PointLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_channels, hidden_size, num_layers, 
                            batch_first=True)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1) 
        
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        
        return self.fc(out)

In [33]:
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_channels = X_train_t.shape[1]

model = PointLSTM(input_channels, hidden_size=32, num_layers=1).to(device)

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
criterion = nn.L1Loss()
early_stopping = EarlyStopping(patience=15, verbose=True) 

X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)

for epoch in range(200):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad() 
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_mae = criterion(val_preds, y_val_t).item()
    
    early_stopping(val_mae, model)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss {total_loss/len(train_loader):.4f}, Val MAE {val_mae:.4f}")
    
    if early_stopping.early_stop:
        break

model.load_state_dict(early_stopping.best_state_dict)
model.eval()

with torch.no_grad():
    final_preds_tensor = model(X_val_t)
    y_true = y_val_t.cpu().numpy().flatten()
    y_pred = final_preds_tensor.cpu().numpy().flatten()
    
    mae = criterion(final_preds_tensor, y_val_t).item()
    rmse = np.sqrt(F.mse_loss(final_preds_tensor, y_val_t).item())
    
    max_err = max_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

print("\n" + "="*40)
print(f"LSTM Point Result (2041-2050):")
print(f"{'MAE':<15}: {mae:.4f}")
print(f"{'RMSE':<15}: {rmse:.4f}")
print(f"{'Max Error':<15}: {max_err:.4f}")
print(f"{'R-Squared':<15}: {r2:.4f}")
print("="*40)


EarlyStopping counter: 1 out of 15
Epoch 10: Train Loss 1.5380, Val MAE 0.8610
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
Epoch 20: Train Loss 1.1882, Val MAE 0.6681
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 6 out of 15
EarlyStopping counter: 7 out of 15
Epoch 30: Train Loss 1.0595, Val MAE 0.5953
EarlyStopping counter: 8 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 6 out of 15
Epoch 40: Train Loss 0.9832, Val MAE 0.6169
EarlyStopping counter: 7 out of 15
EarlyStopping count